# Semana 6 – Actividad 6: Métricas, preprocesamiento y regularización (TensorFlow Keras)

**Curso:** Deep Learning - Conceptos (601539) · FU / CAD2202023205 / EIAIPA2026 · **REA 1**

Se compara un **modelo base** (sin regularización) frente a un **modelo regularizado** (**L2** en pesos de capas densas + **dropout**), con **igual** arquitectura (ancho de capas), **mismos** datos, partición, preprocesamiento, optimizador, épocas y semilla. Se documenta **preprocesamiento** (`StandardScaler`), **métricas** en Keras (accuracy, AUC) y **informe** de clasificación en test (`precision`, `recall`, `F1`).

**Enlaces (tras `git push`):**
- Carpeta **week6/**: [github.com/SonicWD/deep_learning/tree/main/week6](https://github.com/SonicWD/deep_learning/tree/main/week6)
- **Google Colab**: [Abrir notebook](https://colab.research.google.com/github/SonicWD/deep_learning/blob/main/week6/actividad6/Actividad6_Metricas_Preprocesamiento_Regularizacion_Keras.ipynb)


### Ejecución recomendada

**Google Colab:** *Entorno → Ejecutar todo*. En local, use **Python 3.11–3.12** si TensorFlow no instala en versiones muy nuevas.


In [1]:
import subprocess
import sys


def ensure_deps():
    """Intenta cargar TensorFlow y, si falla, instala dependencias mínimas."""
    try:
        import tensorflow as tf  # noqa: F401

        return
    except Exception:
        print("TensorFlow no está disponible. Intentando instalar dependencias...")

    cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "tensorflow==2.16.1",
        "scikit-learn",
        "matplotlib",
        "numpy",
        "pandas",
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(
            "No fue posible instalar TensorFlow en este entorno. "
            "Use Google Colab o Python 3.11/3.12 en local.\n"
            f"Detalle: {proc.stderr[-600:]}"
        )


ensure_deps()

import tensorflow as tf

print("TensorFlow:", tf.__version__)


## 1. Datos y preprocesamiento

- Dataset sintético **binario** (`make_classification`) con ruido en etiquetas para favorecer **sobreajuste** si la red es capaz.
- Partición **estratificada:** entrenamiento / validación / prueba.
- **Preprocesamiento:** `StandardScaler` ajustado **solo** en train; misma transformación en val y test (*data leakage* evitado).


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)

X, y = make_classification(
    n_samples=2000,
    n_features=32,
    n_informative=16,
    n_redundant=6,
    n_clusters_per_class=2,
    flip_y=0.12,
    class_sep=0.85,
    random_state=RANDOM_STATE,
)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_full
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

y_train = y_train.astype(np.float32)
y_val = y_val.astype(np.float32)
y_test = y_test.astype(np.float32)

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)


## 2. Modelo base (sin regularización)

Misma forma de red que el modelo regularizado: capas densas **ReLU** y salida **sigmoide** (clasificación binaria). **Sin** `kernel_regularizer` ni **dropout**.


In [ ]:
HIDDEN = (96, 48)
EPOCHS = 120
BATCH = 32
LR = 1e-3


def build_baseline():
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(X_train.shape[1],)),
            tf.keras.layers.Dense(HIDDEN[0], activation="relu"),
            tf.keras.layers.Dense(HIDDEN[1], activation="relu"),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ],
        name="baseline_sin_regularizacion",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )
    return model


model_base = build_baseline()
hist_base = model_base.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=0,
)

res_base_val = model_base.evaluate(X_val, y_val, verbose=0, return_dict=True)
res_base_test = model_base.evaluate(X_test, y_test, verbose=0, return_dict=True)
print("BASE — val:", res_base_val)
print("BASE — test:", res_base_test)


## 3. Modelo regularizado (L2 + dropout)

**Regularización aplicada:** penalización **L2** (`kernel_regularizer`) en capas densas ocultas y **dropout** tras cada oculta. Mismos tamaños `HIDDEN`, mismo `Adam`, `lr`, `epochs`, `batch` y datos que el base.


In [ ]:
l2 = tf.keras.regularizers.l2(1e-3)
DROP = 0.35


def build_regularized():
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(X_train.shape[1],)),
            tf.keras.layers.Dense(HIDDEN[0], activation="relu", kernel_regularizer=l2),
            tf.keras.layers.Dropout(DROP),
            tf.keras.layers.Dense(HIDDEN[1], activation="relu", kernel_regularizer=l2),
            tf.keras.layers.Dropout(DROP),
            tf.keras.layers.Dense(1, activation="sigmoid"),
        ],
        name="regularizado_L2_dropout",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )
    return model


model_reg = build_regularized()
hist_reg = model_reg.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=0,
)

res_reg_val = model_reg.evaluate(X_val, y_val, verbose=0, return_dict=True)
res_reg_test = model_reg.evaluate(X_test, y_test, verbose=0, return_dict=True)
print("REG  — val:", res_reg_val)
print("REG  — test:", res_reg_test)


## 4. Gráficos: entrenamiento y generalización

Curvas de **loss** y **AUC** (train vs validation) para cada modelo. Una brecha grande val-loss >> train-loss o val-AUC << train-AUC sugiere **overfitting**.


In [ ]:
def plot_histories(h1, title1, h2, title2):
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    ax[0, 0].plot(h1.history["loss"], label="train")
    ax[0, 0].plot(h1.history["val_loss"], label="val")
    ax[0, 0].set_title(f"{title1} — loss")
    ax[0, 0].set_xlabel("Época")
    ax[0, 0].legend()
    ax[0, 1].plot(h1.history["auc"], label="train")
    ax[0, 1].plot(h1.history["val_auc"], label="val")
    ax[0, 1].set_title(f"{title1} — AUC")
    ax[0, 1].set_xlabel("Época")
    ax[0, 1].legend()
    ax[1, 0].plot(h2.history["loss"], label="train")
    ax[1, 0].plot(h2.history["val_loss"], label="val")
    ax[1, 0].set_title(f"{title2} — loss")
    ax[1, 0].set_xlabel("Época")
    ax[1, 0].legend()
    ax[1, 1].plot(h2.history["auc"], label="train")
    ax[1, 1].plot(h2.history["val_auc"], label="val")
    ax[1, 1].set_title(f"{title2} — AUC")
    ax[1, 1].set_xlabel("Época")
    ax[1, 1].legend()
    plt.tight_layout()
    plt.show()


plot_histories(hist_base, "Modelo base", hist_reg, "Modelo regularizado")


## 5. Tabla de métricas y reporte en test

Comparación numérica en **validación** y **test**. En test se muestra además **precision / recall / F1** por clase (umbrales 0.5 en probabilidad).


In [ ]:
rows = [
    {"modelo": "base", "split": "val", **res_base_val},
    {"modelo": "base", "split": "test", **res_base_test},
    {"modelo": "regularizado", "split": "val", **res_reg_val},
    {"modelo": "regularizado", "split": "test", **res_reg_test},
]
df = pd.DataFrame(rows)
try:
    from IPython.display import display

    display(df)
except Exception:
    print(df.to_string(index=False))

for name, m in [("BASE", model_base), ("REG", model_reg)]:
    proba = m.predict(X_test, verbose=0).ravel()
    pred = (proba >= 0.5).astype(int)
    print(f"\n=== {name} — classification_report (test) ===")
    print(classification_report(y_test.astype(int), pred, digits=4))
    print(f"{name} — ROC-AUC (sklearn, test): {roc_auc_score(y_test, proba):.4f}")

gap_base = hist_base.history["accuracy"][-1] - hist_base.history["val_accuracy"][-1]
gap_reg = hist_reg.history["accuracy"][-1] - hist_reg.history["val_accuracy"][-1]
print(f"\nBrecha train-val accuracy (última época) — base: {gap_base:.4f} | regularizado: {gap_reg:.4f}")


## 6. Conclusiones (Markdown — consigna)

Redacte **varias frases** apoyándose en las curvas y en la tabla.

### (i) Evidencia de overfitting / underfitting

- Compare **train vs val** en loss y AUC del **modelo base**. ¿La brecha es mayor que en el modelo regularizado?
- ¿Algún modelo muestra **underfitting** (train y val mal a la vez)?

### (ii) Efecto de la regularización en la generalización

- ¿Mejora **val_auc** o **test_auc** (o reduce **val_loss**) el modelo regularizado frente al base?
- ¿Cambia el reporte **precision/recall/F1** en test?

### (iii) Trade-offs

- **L2 + dropout** pueden **reducir** el ajuste a train: ¿aceptable si mejora val/test?
- ¿Curvas más **estables** o entrenamiento **más lento** en el modelo regularizado?

---
*Comparación válida: misma profundidad y anchos de capa, mismos datos y preprocesamiento, mismos hiperparámetros de entrenamiento salvo términos de regularización.*
